# Synthetic Comcast-Like RxMER Anomaly Baselines (PyTorch)

This notebook creates **fully synthetic** DOCSIS-like RxMER data, trains two baseline model families, and compares them under live-event traffic regimes.

- Public-tied baseline: compact 1-D CNN on static spectral snapshots
- Speculative baseline: compact CNN+GRU+attention on short temporal sequences

No private Comcast data is used.

## Environment

Install dependencies in your notebook kernel environment if needed:

```bash
pip install torch numpy matplotlib pandas scikit-learn
```

In [ ]:
# 0. Setup
import copy
import json
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
NUM_BINS = 256
TIME_STEPS = 16

V2_CLASS_NAMES = [
    "normal",
    "narrowband_ingress",
    "cpd",
    "micro_reflection",
    "amp_compression",
    "impulse_noise",
    "unknown_other",
]
V2_REGIMES = ["pre_event", "event_peak", "post_event"]
V2_SIGNAL_DOMAINS = ["downstream_rxmer", "upstream_return"]
V2_NODE_CONTEXT_FEATURES = [
    "utilization",
    "snr_margin",
    "split_mhz",
    "total_amps_in_node",
    "amps_in_series_on_leg",
    "leg_isolated",
    "flat_loss_db",
    "amp_nf_db",
    "amp_cin_db",
    "tcp_headroom_db",
    "hour_of_day_sin",
    "hour_of_day_cos",
]
V2_EDGE_TARGETS = {
    "max_params": 200_000,
    "p95_latency_ms_cpu_proxy": 5.0,
    "quantization_ready": True,
}
V2_ACCEPTANCE_TARGETS = {
    "macro_f1_min": 0.70,
    "event_peak_macro_f1_delta_vs_public_min": 0.03,
    "edge_targets_required": True,
}

CLASS_NAMES = V2_CLASS_NAMES
REGIMES = V2_REGIMES
NUM_CLASSES = len(CLASS_NAMES)

# Default target sizing for the first full run.
TRAIN_SAMPLES = 40_000
VAL_SAMPLES = 8_000
TEST_SAMPLES = 8_000

# Set to True for quick smoke runs while iterating on notebook logic.
FAST_DEV_MODE = False
if FAST_DEV_MODE:
    TRAIN_SAMPLES = 8_000
    VAL_SAMPLES = 1_600
    TEST_SAMPLES = 1_600

MAX_EPOCHS_V2 = 12
PATIENCE = 3
BATCH_SIZE_V2 = 256
LR = 1e-3
WEIGHT_DECAY = 1e-4

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

NOTEBOOK_DIR = Path('/Users/micahsheller/git/flower/examples/research/comcast-anomaly')
BASE_DIR = NOTEBOOK_DIR / 'artifacts'
DATA_DIR = BASE_DIR / 'data'
CKPT_DIR = BASE_DIR / 'checkpoints'
METRICS_DIR = BASE_DIR / 'metrics'
for d in [DATA_DIR, CKPT_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
print(f'FAST_DEV_MODE={FAST_DEV_MODE}')
print(f'Samples train/val/test: {TRAIN_SAMPLES}/{VAL_SAMPLES}/{TEST_SAMPLES}')
print(f'V2 classes ({NUM_CLASSES}): {CLASS_NAMES}')


## 1. Synthetic Signal Generator

In [ ]:
freq = np.linspace(0.0, 1.0, NUM_BINS, dtype=np.float32)
time_axis = np.linspace(0.0, 1.0, TIME_STEPS, dtype=np.float32)

regime_mix = np.array([0.30, 0.45, 0.25], dtype=np.float32)
regime_util_mean = np.array([0.58, 0.92, 0.70], dtype=np.float32)
regime_snr_shift = np.array([0.0, -1.8, -0.8], dtype=np.float32)

# Domain-specific prevalence profiles by regime.
V2_CLASS_PROBS_BY_DOMAIN = {
    "downstream_rxmer": np.array([
        [0.64, 0.10, 0.08, 0.07, 0.06, 0.03, 0.02],  # pre_event
        [0.38, 0.16, 0.12, 0.10, 0.09, 0.08, 0.07],  # event_peak
        [0.54, 0.12, 0.10, 0.08, 0.07, 0.05, 0.04],  # post_event
    ], dtype=np.float32),
    "upstream_return": np.array([
        [0.56, 0.12, 0.10, 0.05, 0.05, 0.07, 0.05],
        [0.28, 0.19, 0.16, 0.07, 0.07, 0.13, 0.10],
        [0.45, 0.14, 0.12, 0.06, 0.06, 0.10, 0.07],
    ], dtype=np.float32),
}


def _sample_hour_by_regime(regime_ids: np.ndarray) -> np.ndarray:
    # Simple traffic cycle prior: event_peak skewed to evening hours.
    hours = np.zeros_like(regime_ids, dtype=np.float32)
    for r in range(len(REGIMES)):
        idx = np.where(regime_ids == r)[0]
        if idx.size == 0:
            continue
        if r == 0:
            h = np.random.normal(10.0, 3.0, size=idx.size)
        elif r == 1:
            h = np.random.normal(19.0, 2.5, size=idx.size)
        else:
            h = np.random.normal(14.0, 3.5, size=idx.size)
        hours[idx] = np.mod(h, 24.0)
    return hours.astype(np.float32)


def sample_v2_labels_and_meta(n_samples: int, signal_domain: str) -> dict:
    if signal_domain not in V2_SIGNAL_DOMAINS:
        raise ValueError(f"Unsupported signal_domain={signal_domain}")

    regime_ids = np.random.choice(len(REGIMES), size=n_samples, p=regime_mix).astype(np.int64)
    y = np.zeros(n_samples, dtype=np.int64)
    class_probs = V2_CLASS_PROBS_BY_DOMAIN[signal_domain]
    for r in range(len(REGIMES)):
        idx = np.where(regime_ids == r)[0]
        if idx.size > 0:
            y[idx] = np.random.choice(NUM_CLASSES, size=idx.size, p=class_probs[r]).astype(np.int64)

    utilization = np.clip(
        np.random.normal(loc=regime_util_mean[regime_ids], scale=0.05, size=n_samples),
        0.20,
        0.99,
    ).astype(np.float32)

    total_amps_in_node = np.clip(
        np.round(np.random.normal(loc=np.array([24, 46, 34], dtype=np.float32)[regime_ids], scale=5.0)),
        8,
        96,
    ).astype(np.float32)

    amps_in_series_on_leg = np.clip(
        np.round(np.random.normal(loc=np.array([5, 9, 7], dtype=np.float32)[regime_ids], scale=1.3)),
        2,
        16,
    ).astype(np.float32)

    leg_isolated = np.random.binomial(
        1,
        np.array([0.72, 0.52, 0.64], dtype=np.float32)[regime_ids],
        size=n_samples,
    ).astype(np.float32)

    flat_loss_db = np.clip(
        np.random.normal(loc=np.array([1.8, 4.1, 2.8], dtype=np.float32)[regime_ids], scale=0.9, size=n_samples),
        0.0,
        9.0,
    ).astype(np.float32)

    amp_nf_db = np.clip(np.random.normal(6.0, 0.55, size=n_samples), 4.5, 8.5).astype(np.float32)
    amp_cin_db = np.clip(np.random.normal(56.0, 2.5, size=n_samples), 48.0, 64.0).astype(np.float32)

    split_mhz = np.random.choice(
        np.array([204.0, 396.0, 492.0], dtype=np.float32),
        size=n_samples,
        p=np.array([
            0.65 if signal_domain == "downstream_rxmer" else 0.35,
            0.25 if signal_domain == "downstream_rxmer" else 0.35,
            0.10 if signal_domain == "downstream_rxmer" else 0.30,
        ], dtype=np.float32),
    ).astype(np.float32)

    tcp_headroom_db = np.clip(
        np.random.normal(
            loc=np.array([4.8, 2.5, 3.6], dtype=np.float32)[regime_ids]
            - 0.08 * np.maximum(0.0, amps_in_series_on_leg - 6.0),
            scale=0.85,
            size=n_samples,
        ),
        0.0,
        8.0,
    ).astype(np.float32)

    hour = _sample_hour_by_regime(regime_ids)
    hour_of_day_sin = np.sin(2.0 * np.pi * hour / 24.0).astype(np.float32)
    hour_of_day_cos = np.cos(2.0 * np.pi * hour / 24.0).astype(np.float32)

    domain_base = 35.8 if signal_domain == "downstream_rxmer" else 33.0
    snr_margin = (
        domain_base
        + regime_snr_shift[regime_ids]
        - 2.5 * (utilization - 0.5)
        - 0.045 * np.maximum(0.0, total_amps_in_node - 20.0)
        - 0.25 * flat_loss_db
        + 0.15 * leg_isolated
        + np.random.normal(0.0, 0.9, size=n_samples)
    ).astype(np.float32)

    return {
        "signal_domain": signal_domain,
        "y": y,
        "regime_ids": regime_ids,
        "utilization": utilization,
        "snr_margin": snr_margin,
        "split_mhz": split_mhz,
        "total_amps_in_node": total_amps_in_node,
        "amps_in_series_on_leg": amps_in_series_on_leg,
        "leg_isolated": leg_isolated,
        "flat_loss_db": flat_loss_db,
        "amp_nf_db": amp_nf_db,
        "amp_cin_db": amp_cin_db,
        "tcp_headroom_db": tcp_headroom_db,
        "hour_of_day_sin": hour_of_day_sin,
        "hour_of_day_cos": hour_of_day_cos,
    }


def build_v2_node_context(meta: dict) -> np.ndarray:
    cols = []
    for key in V2_NODE_CONTEXT_FEATURES:
        if key not in meta:
            raise KeyError(f"Missing context feature: {key}")
        cols.append(meta[key].astype(np.float32))
    return np.stack(cols, axis=1).astype(np.float32)


def build_v2_static_profiles(meta: dict) -> tuple[np.ndarray, dict]:
    y = meta["y"]
    regime_ids = meta["regime_ids"]
    utilization = meta["utilization"]
    snr_margin = meta["snr_margin"]
    domain = meta["signal_domain"]

    n = y.shape[0]
    tilt = np.random.uniform(-1.4, 1.4, size=n).astype(np.float32)

    base = (
        snr_margin[:, None]
        + 0.85 * np.cos(2.0 * np.pi * 1.4 * freq)[None, :]
        + 0.30 * np.sin(2.0 * np.pi * 0.55 * freq)[None, :]
        + tilt[:, None] * (freq[None, :] - 0.5)
    ).astype(np.float32)

    if domain == "upstream_return":
        # Extra stress from return-path topology and losses.
        base -= 0.030 * np.maximum(0.0, meta["total_amps_in_node"] - 20.0)[:, None]
        base -= 0.20 * meta["flat_loss_db"][:, None]
        base += 0.18 * meta["leg_isolated"][:, None]

    impair = np.zeros_like(base, dtype=np.float32)
    latent = {
        "ingress_center": np.zeros(n, dtype=np.float32),
        "ingress_width": np.zeros(n, dtype=np.float32),
        "ingress_depth": np.zeros(n, dtype=np.float32),
        "cpd_freq": np.zeros(n, dtype=np.float32),
        "cpd_phase": np.zeros(n, dtype=np.float32),
        "cpd_amp": np.zeros(n, dtype=np.float32),
        "micro_freq": np.zeros(n, dtype=np.float32),
        "micro_phase": np.zeros(n, dtype=np.float32),
        "micro_amp": np.zeros(n, dtype=np.float32),
        "comp_strength": np.zeros(n, dtype=np.float32),
        "impulse_bins": np.full((n, 8), -1, dtype=np.int64),
        "impulse_amp": np.zeros((n, 8), dtype=np.float32),
        "unknown_mode": np.zeros(n, dtype=np.int64),
    }

    ingress_idx = np.where(y == 1)[0]
    if ingress_idx.size > 0:
        c = np.random.uniform(0.07, 0.93, size=ingress_idx.size).astype(np.float32)
        w = np.random.uniform(0.010, 0.040, size=ingress_idx.size).astype(np.float32)
        d = np.random.uniform(2.8, 10.5, size=ingress_idx.size).astype(np.float32)
        notch = -d[:, None] * np.exp(-0.5 * ((freq[None, :] - c[:, None]) / w[:, None]) ** 2)
        impair[ingress_idx] += notch.astype(np.float32)
        latent["ingress_center"][ingress_idx] = c
        latent["ingress_width"][ingress_idx] = w
        latent["ingress_depth"][ingress_idx] = d

    cpd_idx = np.where(y == 2)[0]
    if cpd_idx.size > 0:
        h = np.random.uniform(16.0, 42.0, size=cpd_idx.size).astype(np.float32)
        p = np.random.uniform(0.0, 2.0 * np.pi, size=cpd_idx.size).astype(np.float32)
        a = np.random.uniform(1.6, 5.2, size=cpd_idx.size).astype(np.float32)
        comb = -a[:, None] * (0.5 + 0.5 * np.sin(2.0 * np.pi * h[:, None] * freq[None, :] + p[:, None])) ** 2
        impair[cpd_idx] += comb.astype(np.float32)
        latent["cpd_freq"][cpd_idx] = h
        latent["cpd_phase"][cpd_idx] = p
        latent["cpd_amp"][cpd_idx] = a

    micro_idx = np.where(y == 3)[0]
    if micro_idx.size > 0:
        rf = np.random.uniform(6.0, 18.0, size=micro_idx.size).astype(np.float32)
        rp = np.random.uniform(0.0, 2.0 * np.pi, size=micro_idx.size).astype(np.float32)
        ra = np.random.uniform(1.0, 3.6, size=micro_idx.size).astype(np.float32)
        ripple = -ra[:, None] * np.sin(2.0 * np.pi * rf[:, None] * freq[None, :] + rp[:, None])
        impair[micro_idx] += ripple.astype(np.float32)
        latent["micro_freq"][micro_idx] = rf
        latent["micro_phase"][micro_idx] = rp
        latent["micro_amp"][micro_idx] = ra

    comp_idx = np.where(y == 4)[0]
    if comp_idx.size > 0:
        s = np.random.uniform(2.2, 5.8, size=comp_idx.size).astype(np.float32)
        shaping = (0.56 + 0.78 * (freq[None, :] ** 1.6)).astype(np.float32)
        comp_drop = -s[:, None] * shaping
        impair[comp_idx] += comp_drop
        latent["comp_strength"][comp_idx] = s

    impulse_idx = np.where(y == 5)[0]
    if impulse_idx.size > 0:
        for ii in impulse_idx:
            k = np.random.randint(3, 9)
            bins = np.random.choice(NUM_BINS, size=k, replace=False)
            amps = np.random.uniform(2.8, 8.0, size=k).astype(np.float32)
            for j, b in enumerate(bins):
                impair[ii, b] -= amps[j]
                latent["impulse_bins"][ii, j] = int(b)
                latent["impulse_amp"][ii, j] = amps[j]

    unknown_idx = np.where(y == 6)[0]
    if unknown_idx.size > 0:
        modes = np.random.choice(4, size=unknown_idx.size).astype(np.int64)
        latent["unknown_mode"][unknown_idx] = modes
        for loc, mode in zip(unknown_idx, modes):
            if mode == 0:
                # Dual notch signature.
                c = np.random.uniform([0.15, 0.55], [0.40, 0.90]).astype(np.float32)
                w = np.random.uniform(0.010, 0.035, size=2).astype(np.float32)
                d = np.random.uniform(2.0, 6.5, size=2).astype(np.float32)
                for cc, ww, dd in zip(c, w, d):
                    impair[loc] += -dd * np.exp(-0.5 * ((freq - cc) / ww) ** 2)
            elif mode == 1:
                # Chirp-like frequency perturbation.
                impair[loc] += -2.0 * np.sin(2.0 * np.pi * (5.0 + 9.0 * freq) * freq + np.random.uniform(0, 2*np.pi))
            elif mode == 2:
                # Piecewise slope break.
                pivot = np.random.uniform(0.25, 0.75)
                left = -2.2 * (freq - pivot)
                right = -5.0 * np.maximum(0.0, freq - pivot)
                impair[loc] += left + right
            else:
                # Hybrid ripple + sparse dips.
                impair[loc] += -1.2 * np.sin(2.0 * np.pi * np.random.uniform(8.0, 16.0) * freq + np.random.uniform(0, 2*np.pi))
                bins = np.random.choice(NUM_BINS, size=np.random.randint(3, 7), replace=False)
                impair[loc, bins] -= np.random.uniform(1.5, 4.5, size=bins.size).astype(np.float32)

    static = base + impair
    regime_noise = np.array([0.45, 0.95, 0.68], dtype=np.float32)
    if domain == "upstream_return":
        regime_noise = regime_noise + 0.30
    static += np.random.normal(0.0, regime_noise[regime_ids][:, None], size=static.shape).astype(np.float32)

    if domain == "downstream_rxmer":
        static = np.clip(static, 16.0, 45.0).astype(np.float32)
    else:
        static = np.clip(static, 10.0, 42.0).astype(np.float32)

    return static, latent


def build_v2_sequence_profiles(x_static: np.ndarray, meta: dict, latent: dict) -> np.ndarray:
    y = meta["y"]
    regime_ids = meta["regime_ids"]
    domain = meta["signal_domain"]

    n = x_static.shape[0]
    seq = np.repeat(x_static[:, None, :], TIME_STEPS, axis=1).astype(np.float32)

    # Global temporal drift driven by utilization and split profile.
    util = meta["utilization"]
    split_scale = np.clip((meta["split_mhz"] - 204.0) / 288.0, 0.0, 1.0).astype(np.float32)
    drift_amp = (0.18 + 0.45 * util + 0.25 * split_scale).astype(np.float32)
    global_wave = np.sin(2.0 * np.pi * time_axis)[None, :, None].astype(np.float32)
    freq_tilt = (freq[None, None, :] - 0.5).astype(np.float32)
    seq += drift_amp[:, None, None] * global_wave * freq_tilt

    ingress_idx = np.where(y == 1)[0]
    if ingress_idx.size > 0:
        c = latent["ingress_center"][ingress_idx]
        w = np.maximum(latent["ingress_width"][ingress_idx], 1e-3)
        d = latent["ingress_depth"][ingress_idx]
        ph = np.random.uniform(0.0, 2.0 * np.pi, size=ingress_idx.size).astype(np.float32)
        c_t = c[:, None] + 0.020 * np.sin(2.0 * np.pi * 1.8 * time_axis[None, :] + ph[:, None])
        d_t = d[:, None] * (1.0 + 0.18 * np.sin(2.0 * np.pi * 2.6 * time_axis[None, :] + ph[:, None]))
        notch = -d_t[:, :, None] * np.exp(
            -0.5 * ((freq[None, None, :] - c_t[:, :, None]) / w[:, None, None]) ** 2
        )
        seq[ingress_idx] += notch.astype(np.float32)

    cpd_idx = np.where(y == 2)[0]
    if cpd_idx.size > 0:
        h = latent["cpd_freq"][cpd_idx]
        p = latent["cpd_phase"][cpd_idx]
        a = latent["cpd_amp"][cpd_idx]
        amp_t = a[:, None] * (1.0 + 0.24 * np.sin(2.0 * np.pi * 3.2 * time_axis[None, :] + p[:, None]))
        comb = -amp_t[:, :, None] * (
            0.5 + 0.5 * np.sin(2.0 * np.pi * h[:, None, None] * freq[None, None, :] + p[:, None, None])
        ) ** 2
        seq[cpd_idx] += comb.astype(np.float32)

    micro_idx = np.where(y == 3)[0]
    if micro_idx.size > 0:
        rf = latent["micro_freq"][micro_idx]
        rp = latent["micro_phase"][micro_idx]
        ra = latent["micro_amp"][micro_idx]
        phase_t = rp[:, None] + 2.0 * np.pi * 0.20 * time_axis[None, :]
        ripple = -ra[:, None, None] * np.sin(
            2.0 * np.pi * rf[:, None, None] * freq[None, None, :] + phase_t[:, :, None]
        )
        seq[micro_idx] += ripple.astype(np.float32)

    comp_idx = np.where(y == 4)[0]
    if comp_idx.size > 0:
        s = latent["comp_strength"][comp_idx]
        burst = (1.0 + 0.33 * np.exp(-0.5 * ((time_axis - 0.72) / 0.17) ** 2)).astype(np.float32)
        shaping = (0.55 + 0.80 * (freq ** 1.6)).astype(np.float32)
        seq[comp_idx] += (-s[:, None, None] * burst[None, :, None] * shaping[None, None, :]).astype(np.float32)

    impulse_idx = np.where(y == 5)[0]
    if impulse_idx.size > 0:
        for ii in impulse_idx:
            burst_count = np.random.randint(2, 6)
            t_idx = np.random.choice(TIME_STEPS, size=burst_count, replace=False)
            bins = latent["impulse_bins"][ii]
            amps = latent["impulse_amp"][ii]
            valid = np.where(bins >= 0)[0]
            if valid.size == 0:
                continue
            for tt in t_idx:
                jitter = np.random.uniform(0.8, 1.2)
                seq[ii, tt, bins[valid]] -= jitter * amps[valid]

    unknown_idx = np.where(y == 6)[0]
    if unknown_idx.size > 0:
        for ii in unknown_idx:
            mode = latent["unknown_mode"][ii]
            if mode == 0:
                # Alternating dual notch dynamics.
                center = np.random.uniform(0.18, 0.82)
                width = np.random.uniform(0.01, 0.03)
                for t in range(TIME_STEPS):
                    shift = 0.025 * np.sin(2 * np.pi * t / TIME_STEPS)
                    depth = 2.5 + 1.2 * np.cos(2 * np.pi * 2 * t / TIME_STEPS)
                    seq[ii, t] += -depth * np.exp(-0.5 * ((freq - (center + shift)) / width) ** 2)
            elif mode == 1:
                # Irregular comb/ripple crossover.
                ph = np.random.uniform(0, 2*np.pi)
                for t in range(TIME_STEPS):
                    comb = -1.5 * (0.5 + 0.5 * np.sin(2*np.pi*(12 + 6*np.sin(ph + t*0.3))*freq + ph))**2
                    seq[ii, t] += comb.astype(np.float32)
            elif mode == 2:
                # Burst-on-slope behavior.
                slope = -2.8 * (freq - 0.4)
                seq[ii] += slope[None, :]
                for _ in range(np.random.randint(2, 5)):
                    t0 = np.random.randint(0, TIME_STEPS)
                    b0 = np.random.randint(0, NUM_BINS - 4)
                    seq[ii, t0, b0:b0+4] -= np.random.uniform(1.5, 4.5)
            else:
                # Low-rank random perturbation.
                a = np.random.normal(0.0, 1.0, size=TIME_STEPS).astype(np.float32)
                b = np.random.normal(0.0, 1.0, size=NUM_BINS).astype(np.float32)
                seq[ii] += 0.45 * np.outer(a, b).astype(np.float32)

    temporal_sigma = np.array([0.26, 0.68, 0.40], dtype=np.float32)[regime_ids]
    if domain == "upstream_return":
        temporal_sigma += 0.18
    seq += np.random.normal(0.0, temporal_sigma[:, None, None], size=seq.shape).astype(np.float32)

    if domain == "downstream_rxmer":
        seq = np.clip(seq, 16.0, 45.0).astype(np.float32)
    else:
        seq = np.clip(seq, 10.0, 42.0).astype(np.float32)

    return seq


# Bind implementation aliases used by scaffolded signatures.
sample_v2_labels_and_meta_impl = sample_v2_labels_and_meta
build_v2_static_profiles_impl = build_v2_static_profiles
build_v2_sequence_profiles_impl = build_v2_sequence_profiles
build_v2_node_context_impl = build_v2_node_context


## 2. Dataset Assembly and Validation

In [ ]:
def _build_domain_dataset(signal_domain: str, train_samples: int, val_samples: int, test_samples: int):
    total_samples = train_samples + val_samples + test_samples
    print(f"[{signal_domain}] Generating {total_samples:,} samples...")
    t0 = time.time()

    meta_all = sample_v2_labels_and_meta(total_samples, signal_domain)
    X_static_all, latent_params = build_v2_static_profiles(meta_all)
    X_seq_all = build_v2_sequence_profiles(X_static_all, meta_all, latent_params)
    X_context_all = build_v2_node_context(meta_all)

    gen_seconds = time.time() - t0
    print(f"[{signal_domain}] Generation finished in {gen_seconds:.1f}s")

    y_all = meta_all["y"]
    regime_ids_all = meta_all["regime_ids"]
    strata_all = (y_all * len(REGIMES) + regime_ids_all).astype(np.int64)
    all_indices = np.arange(total_samples)

    train_idx, temp_idx = train_test_split(
        all_indices,
        train_size=train_samples,
        random_state=SEED,
        stratify=strata_all,
    )

    temp_strata = strata_all[temp_idx]
    val_fraction_of_temp = val_samples / (val_samples + test_samples)
    val_idx, test_idx = train_test_split(
        temp_idx,
        train_size=val_fraction_of_temp,
        random_state=SEED,
        stratify=temp_strata,
    )

    def split_dict(idx):
        out = {
            "X_static": X_static_all[idx],
            "X_seq": X_seq_all[idx],
            "X_context": X_context_all[idx],
            "y": y_all[idx],
            "regime_ids": regime_ids_all[idx],
        }
        for k in V2_NODE_CONTEXT_FEATURES:
            out[k] = meta_all[k][idx]
        return out

    data = {
        "signal_domain": signal_domain,
        "gen_seconds": float(gen_seconds),
        "all": {
            "X_static": X_static_all,
            "X_seq": X_seq_all,
            "X_context": X_context_all,
            "y": y_all,
            "regime_ids": regime_ids_all,
        },
        "train": split_dict(train_idx),
        "val": split_dict(val_idx),
        "test": split_dict(test_idx),
    }

    # Validation checks.
    assert data["train"]["X_static"].shape == (train_samples, NUM_BINS)
    assert data["val"]["X_static"].shape == (val_samples, NUM_BINS)
    assert data["test"]["X_static"].shape == (test_samples, NUM_BINS)

    assert data["train"]["X_seq"].shape == (train_samples, TIME_STEPS, NUM_BINS)
    assert data["val"]["X_seq"].shape == (val_samples, TIME_STEPS, NUM_BINS)
    assert data["test"]["X_seq"].shape == (test_samples, TIME_STEPS, NUM_BINS)

    assert data["train"]["X_context"].shape == (train_samples, len(V2_NODE_CONTEXT_FEATURES))
    assert data["val"]["X_context"].shape == (val_samples, len(V2_NODE_CONTEXT_FEATURES))
    assert data["test"]["X_context"].shape == (test_samples, len(V2_NODE_CONTEXT_FEATURES))

    for split_name in ["train", "val", "test"]:
        for arr_key in ["X_static", "X_seq", "X_context"]:
            arr = data[split_name][arr_key]
            assert np.isfinite(arr).all(), f"non-finite values in {signal_domain}/{split_name}/{arr_key}"

        y_split = data[split_name]["y"]
        regime_split = data[split_name]["regime_ids"]
        assert set(np.unique(y_split).tolist()) == set(range(NUM_CLASSES)), f"missing class in {signal_domain}/{split_name}"
        assert set(np.unique(regime_split).tolist()) == set(range(len(REGIMES))), f"missing regime in {signal_domain}/{split_name}"

    print(
        f"[{signal_domain}] train class counts:",
        dict(zip(CLASS_NAMES, np.bincount(data['train']['y'], minlength=NUM_CLASSES))),
    )
    print(
        f"[{signal_domain}] train regime counts:",
        dict(zip(REGIMES, np.bincount(data['train']['regime_ids'], minlength=len(REGIMES)))),
    )

    return data


DOMAIN_DATA = {
    domain: _build_domain_dataset(domain, TRAIN_SAMPLES, VAL_SAMPLES, TEST_SAMPLES)
    for domain in V2_SIGNAL_DOMAINS
}

v2_generation_seconds = {d: DOMAIN_DATA[d]["gen_seconds"] for d in V2_SIGNAL_DOMAINS}
print("Generation seconds by domain:", v2_generation_seconds)


In [ ]:
# Persist full synthetic v2 dataset contract
dataset_path = DATA_DIR / "comcast_synthetic_rxmer_v2.npz"

save_map = {}
for domain, prefix in [("downstream_rxmer", "ds"), ("upstream_return", "us")]:
    d = DOMAIN_DATA[domain]
    for split in ["train", "val", "test"]:
        save_map[f"{prefix}_X_static_{split}"] = d[split]["X_static"]
        save_map[f"{prefix}_X_seq_{split}"] = d[split]["X_seq"]
        save_map[f"{prefix}_X_context_{split}"] = d[split]["X_context"]
        save_map[f"{prefix}_y_{split}"] = d[split]["y"]
        save_map[f"{prefix}_regime_{split}"] = d[split]["regime_ids"]
        for feat in V2_NODE_CONTEXT_FEATURES:
            save_map[f"{prefix}_{feat}_{split}"] = d[split][feat]

np.savez_compressed(dataset_path, **save_map)
print(f"Saved dataset: {dataset_path.resolve()}")


In [ ]:
# Domain-aware sanity: one representative static sample per class for each domain
fig, axes = plt.subplots(len(V2_SIGNAL_DOMAINS), NUM_CLASSES, figsize=(2.6 * NUM_CLASSES, 2.8 * len(V2_SIGNAL_DOMAINS)), sharex=True)
for r, domain in enumerate(V2_SIGNAL_DOMAINS):
    y_train = DOMAIN_DATA[domain]["train"]["y"]
    X_train = DOMAIN_DATA[domain]["train"]["X_static"]
    for c in range(NUM_CLASSES):
        idx = np.where(y_train == c)[0][0]
        ax = axes[r, c] if len(V2_SIGNAL_DOMAINS) > 1 else axes[c]
        ax.plot(X_train[idx], linewidth=1.0)
        if r == 0:
            ax.set_title(CLASS_NAMES[c], fontsize=9)
        if c == 0:
            ax.set_ylabel(f"{domain}\nMER dB", fontsize=8)
        ax.grid(alpha=0.15)

axes[-1, 0].set_xlabel("Subcarrier / Bin")
plt.tight_layout()
plt.show()


## 2b. Visual Anomaly Checklist (Annotated)

These figures are a quick visual guide for spotting each synthetic signature by eye.


### Overview

![Annotated checklist overview](artifacts/figures/checklist/checklist_overview_annotated_v2.png)

### Individual Signatures

- **Normal**: broad baseline shape with only small random variation.
![Normal annotated](artifacts/figures/checklist/01_normal_gentle_shape_annotated.png)

- **Narrowband ingress**: one deep, localized notch.
![Single deep notch annotated](artifacts/figures/checklist/02_single_deep_notch_annotated.png)

- **CPD-like**: many fine repeating dips (comb texture) across the band.
![Fine repeating comb dips annotated](artifacts/figures/checklist/03_fine_repeating_comb_dips_annotated.png)

- **Micro-reflection**: smooth sinusoidal ripple with wider spacing.
![Smooth repeating ripple annotated](artifacts/figures/checklist/04_smooth_repeating_ripple_annotated.png)

- **Amp compression**: overall left-to-right MER collapse.
![Left to right collapse annotated](artifacts/figures/checklist/05_left_to_right_collapse_annotated.png)


## 2c. Upstream + Amplifier-Node v2 Plan Scaffold

This section adds a decision-locked scaffold for the next implementation pass.
It does **not** change the current baseline training/evaluation flow in this notebook.


In [ ]:
# v2 taxonomy, context schema, and deployment gates (scaffold constants only)
V2_CLASS_NAMES = [
    "normal",
    "narrowband_ingress",
    "cpd",
    "micro_reflection",
    "amp_compression",
    "impulse_noise",
    "unknown_other",
]

V2_REGIMES = ["pre_event", "event_peak", "post_event"]
V2_SIGNAL_DOMAINS = ["downstream_rxmer", "upstream_return"]

V2_NODE_CONTEXT_FEATURES = [
    "utilization",
    "snr_margin",
    "split_mhz",
    "total_amps_in_node",
    "amps_in_series_on_leg",
    "leg_isolated",
    "flat_loss_db",
    "amp_nf_db",
    "amp_cin_db",
    "tcp_headroom_db",
    "hour_of_day_sin",
    "hour_of_day_cos",
]

V2_EDGE_TARGETS = {
    "max_params": 200_000,
    "p95_latency_ms_cpu_proxy": 5.0,
    "quantization_ready": True,
}

V2_ACCEPTANCE_TARGETS = {
    "macro_f1_min": 0.70,
    "event_peak_macro_f1_delta_vs_public_min": 0.03,
    "edge_targets_required": True,
}

print("V2 classes:", V2_CLASS_NAMES)
print("V2 node context features:", len(V2_NODE_CONTEXT_FEATURES))


In [ ]:
# Persist v2 contract artifact
SPECS_DIR = BASE_DIR / "specs"
SPECS_DIR.mkdir(parents=True, exist_ok=True)

v2_contract = {
    "V2_CLASS_NAMES": V2_CLASS_NAMES,
    "V2_REGIMES": V2_REGIMES,
    "V2_SIGNAL_DOMAINS": V2_SIGNAL_DOMAINS,
    "V2_NODE_CONTEXT_FEATURES": V2_NODE_CONTEXT_FEATURES,
    "V2_EDGE_TARGETS": V2_EDGE_TARGETS,
    "V2_ACCEPTANCE_TARGETS": V2_ACCEPTANCE_TARGETS,
}

v2_contract_path = SPECS_DIR / "upstream_node_v2_contract.json"
with v2_contract_path.open("w", encoding="utf-8") as f:
    json.dump(v2_contract, f, indent=2)

print("Saved v2 contract:", v2_contract_path.resolve())


### Regime and Modeling Assumptions (v2 target)

- `pre_event`: lower utilization and lower impairment pressure.
- `event_peak`: highest utilization and highest impairment/noise pressure.
- `post_event`: partial recovery with intermediate pressure.
- Event stress should be driven by operational variables, not only class priors.
- Upstream funneling assumptions include impulse noise, ingress noise, and CPD relevance.
- Node topology factors (`total amps`, `series amps`, `leg isolation`, `flat loss`) are first-class in v2.


In [ ]:
# v2 function interfaces (implemented)
def sample_v2_labels_and_meta(n_samples: int, signal_domain: str) -> dict:
    return globals()["sample_v2_labels_and_meta_impl"](n_samples, signal_domain)


def build_v2_static_profiles(meta: dict) -> tuple[np.ndarray, dict]:
    return globals()["build_v2_static_profiles_impl"](meta)


def build_v2_sequence_profiles(x_static: np.ndarray, meta: dict, latent: dict) -> np.ndarray:
    return globals()["build_v2_sequence_profiles_impl"](x_static, meta, latent)


def build_v2_node_context(meta: dict) -> np.ndarray:
    return globals()["build_v2_node_context_impl"](meta)


def train_v2_context_model(train_bundle: dict, val_bundle: dict) -> dict:
    return globals()["train_v2_context_model_impl"](train_bundle, val_bundle)


def evaluate_v2(model_bundle: dict, test_bundle: dict) -> dict:
    return globals()["evaluate_v2_impl"](model_bundle, test_bundle)


### Edge-NPU Evaluation Gates (v2 target)

For amplifier-node deployment, model quality alone is insufficient.
v2 should evaluate quality *and* edge constraints together:

- Parameter budget (`max_params`).
- Latency proxy (`p95_latency_ms_cpu_proxy`).
- Quantization readiness (`quantization_ready`).
- Quality metrics (macro-F1, event-peak F1, anomaly AUROC, per-class F1 for impulse/unknown).


In [ ]:
# Empty scaffold table for future edge-qualified model comparison
v2_eval_columns = [
    "model_name",
    "params",
    "p95_latency_ms_cpu_proxy",
    "quantization_ready",
    "macro_f1",
    "event_peak_macro_f1",
    "anomaly_auroc",
    "impulse_f1",
    "unknown_f1",
    "pass_edge_gate",
]

v2_eval_template_df = pd.DataFrame(columns=v2_eval_columns)
v2_eval_template_df


## 3. V2 Data, Normalization, and Sequence+Context Model


In [ ]:
def _normalize_domain_splits(domain_data: dict):
    tr, va, te = domain_data["train"], domain_data["val"], domain_data["test"]

    static_mean = tr["X_static"].mean(axis=0, keepdims=True)
    static_std = tr["X_static"].std(axis=0, keepdims=True) + 1e-6

    seq_mean = tr["X_seq"].mean(axis=(0, 1), keepdims=True)
    seq_std = tr["X_seq"].std(axis=(0, 1), keepdims=True) + 1e-6

    ctx_mean = tr["X_context"].mean(axis=0, keepdims=True)
    ctx_std = tr["X_context"].std(axis=0, keepdims=True) + 1e-6

    def norm(split: dict):
        return {
            "X_static_n": ((split["X_static"] - static_mean) / static_std).astype(np.float32),
            "X_seq_n": ((split["X_seq"] - seq_mean) / seq_std).astype(np.float32),
            "X_context_n": ((split["X_context"] - ctx_mean) / ctx_std).astype(np.float32),
            "y": split["y"].astype(np.int64),
            "regime_ids": split["regime_ids"].astype(np.int64),
        }

    return {
        "train": norm(tr),
        "val": norm(va),
        "test": norm(te),
        "stats": {
            "static_mean": static_mean.astype(np.float32),
            "static_std": static_std.astype(np.float32),
            "seq_mean": seq_mean.astype(np.float32),
            "seq_std": seq_std.astype(np.float32),
            "ctx_mean": ctx_mean.astype(np.float32),
            "ctx_std": ctx_std.astype(np.float32),
        },
    }


class SeqContextFusionModel(nn.Module):
    def __init__(self, num_bins: int, context_dim: int, num_classes: int, hidden_dim: int = 48):
        super().__init__()
        self.step_encoder = nn.Sequential(
            nn.Conv1d(1, 24, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(24, 48, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(48, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.gru = nn.GRU(input_size=64, hidden_size=hidden_dim, batch_first=True)
        self.attn = nn.Linear(hidden_dim, 1)

        self.context_proj = nn.Sequential(
            nn.Linear(context_dim, 32),
            nn.ReLU(),
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_dim + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(64, num_classes),
        )

    def forward(self, x_seq, x_ctx):
        b, t, f = x_seq.shape
        z = x_seq.reshape(b * t, 1, f)
        z = self.step_encoder(z).squeeze(-1)
        z = z.reshape(b, t, -1)
        h, _ = self.gru(z)
        scores = self.attn(h).squeeze(-1)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        seq_context = (h * weights).sum(dim=1)

        ctx = self.context_proj(x_ctx)
        fused = torch.cat([seq_context, ctx], dim=1)
        return self.head(fused)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


DOMAIN_BUNDLES = {}
for domain in V2_SIGNAL_DOMAINS:
    normed = _normalize_domain_splits(DOMAIN_DATA[domain])

    train_ds = TensorDataset(
        torch.from_numpy(normed["train"]["X_seq_n"]),
        torch.from_numpy(normed["train"]["X_context_n"]),
        torch.from_numpy(normed["train"]["y"]),
        torch.from_numpy(normed["train"]["regime_ids"]),
    )
    val_ds = TensorDataset(
        torch.from_numpy(normed["val"]["X_seq_n"]),
        torch.from_numpy(normed["val"]["X_context_n"]),
        torch.from_numpy(normed["val"]["y"]),
        torch.from_numpy(normed["val"]["regime_ids"]),
    )
    test_ds = TensorDataset(
        torch.from_numpy(normed["test"]["X_seq_n"]),
        torch.from_numpy(normed["test"]["X_context_n"]),
        torch.from_numpy(normed["test"]["y"]),
        torch.from_numpy(normed["test"]["regime_ids"]),
    )

    DOMAIN_BUNDLES[domain] = {
        "domain": domain,
        "train_loader": DataLoader(train_ds, batch_size=BATCH_SIZE_V2, shuffle=True, num_workers=0),
        "val_loader": DataLoader(val_ds, batch_size=BATCH_SIZE_V2, shuffle=False, num_workers=0),
        "test_loader": DataLoader(test_ds, batch_size=BATCH_SIZE_V2, shuffle=False, num_workers=0),
        "normed": normed,
        "context_dim": len(V2_NODE_CONTEXT_FEATURES),
    }

print('Domain bundles ready:', {k: {"train": len(v["train_loader"].dataset), "val": len(v["val_loader"].dataset), "test": len(v["test_loader"].dataset)} for k, v in DOMAIN_BUNDLES.items()})


In [ ]:
def _collect_probs(model, loader, device):
    model.eval()
    all_probs, all_true, all_regime = [], [], []
    with torch.no_grad():
        for x_seq, x_ctx, yb, rb in loader:
            logits = model(x_seq.to(device), x_ctx.to(device))
            probs = torch.softmax(logits, dim=1)
            all_probs.append(probs.cpu().numpy())
            all_true.append(yb.numpy())
            all_regime.append(rb.numpy())
    return np.concatenate(all_probs), np.concatenate(all_true), np.concatenate(all_regime)


def _choose_unknown_threshold(probs: np.ndarray, y_true: np.ndarray, unknown_idx: int = 6) -> float:
    score = 1.0 - probs.max(axis=1)
    target = (y_true == unknown_idx).astype(np.int64)
    if target.min() == target.max():
        return 1.0

    grid = np.linspace(0.0, 1.0, 201)
    best_t = 0.5
    best_f1 = -1.0
    for t in grid:
        pred_u = (score >= t).astype(np.int64)
        f1 = f1_score(target, pred_u, zero_division=0)
        if f1 > best_f1:
            best_f1 = float(f1)
            best_t = float(t)
    return best_t


def _metrics_from_preds(y_true, y_pred, probs, regime_ids, class_names, regimes):
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))

    anomaly_target = (y_true != 0).astype(np.int64)
    anomaly_score = 1.0 - probs.max(axis=1)
    if anomaly_target.min() == anomaly_target.max():
        anomaly_auroc = float('nan')
    else:
        anomaly_auroc = float(roc_auc_score(anomaly_target, anomaly_score))

    by_regime = {}
    for r, name in enumerate(regimes):
        idx = np.where(regime_ids == r)[0]
        by_regime[name] = float('nan') if idx.size == 0 else float(f1_score(y_true[idx], y_pred[idx], average='macro'))

    return {
        'macro_f1': float(f1_score(y_true, y_pred, average='macro')),
        'accuracy': float((y_true == y_pred).mean()),
        'anomaly_auroc': anomaly_auroc,
        'event_peak_macro_f1': by_regime.get('event_peak', float('nan')),
        'by_regime_macro_f1': by_regime,
        'classification_report': report,
        'confusion_matrix': cm.tolist(),
        'impulse_f1': float(report.get('impulse_noise', {}).get('f1-score', 0.0)),
        'unknown_f1': float(report.get('unknown_other', {}).get('f1-score', 0.0)),
    }


def _train_model(model, train_loader, val_loader, device, max_epochs, patience, lr, weight_decay):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    model = model.to(device)
    best_val_f1 = -1.0
    best_state = None
    best_epoch = -1
    wait = 0

    train_loss_history = []
    val_f1_history = []

    t0 = time.time()
    for epoch in range(1, max_epochs + 1):
        model.train()
        running_loss, n_seen = 0.0, 0
        for x_seq, x_ctx, yb, _ in train_loader:
            x_seq = x_seq.to(device)
            x_ctx = x_ctx.to(device)
            yb = yb.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x_seq, x_ctx)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            bs = x_seq.shape[0]
            running_loss += loss.item() * bs
            n_seen += bs

        epoch_loss = running_loss / max(1, n_seen)

        val_probs, val_true, _ = _collect_probs(model, val_loader, device)
        val_pred = np.argmax(val_probs, axis=1)
        val_f1 = float(f1_score(val_true, val_pred, average='macro'))

        train_loss_history.append(float(epoch_loss))
        val_f1_history.append(float(val_f1))
        print(f'Epoch {epoch:02d} | train_loss={epoch_loss:.4f} | val_macro_f1={val_f1:.4f}')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f'Early stopping at epoch {epoch} (best epoch {best_epoch})')
                break

    train_seconds = time.time() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    return {
        'model': model,
        'best_epoch': best_epoch,
        'best_val_macro_f1': float(best_val_f1),
        'train_loss_history': train_loss_history,
        'val_f1_history': val_f1_history,
        'train_seconds': float(train_seconds),
    }


def _measure_latency_p95_cpu_ms(model, seq_sample: np.ndarray, ctx_sample: np.ndarray) -> float:
    cpu_model = copy.deepcopy(model).cpu().eval()
    seq_t = torch.from_numpy(seq_sample[None, ...].astype(np.float32))
    ctx_t = torch.from_numpy(ctx_sample[None, ...].astype(np.float32))

    with torch.no_grad():
        for _ in range(50):
            _ = cpu_model(seq_t, ctx_t)

        times = []
        for _ in range(500):
            t0 = time.perf_counter()
            _ = cpu_model(seq_t, ctx_t)
            times.append((time.perf_counter() - t0) * 1000.0)

    return float(np.percentile(np.array(times, dtype=np.float32), 95.0))


def _quantization_readiness(model, seq_sample: np.ndarray, ctx_sample: np.ndarray):
    seq_t = torch.from_numpy(seq_sample[None, ...].astype(np.float32))
    ctx_t = torch.from_numpy(ctx_sample[None, ...].astype(np.float32))

    quantize_dynamic = getattr(torch.quantization, 'quantize_dynamic', None)
    if quantize_dynamic is None:
        try:
            from torch.ao.quantization import quantize_dynamic as qdyn
            quantize_dynamic = qdyn
        except Exception:
            return False, 'quantize_dynamic_not_available'

    try:
        cpu_model = copy.deepcopy(model).cpu().eval()
        qmodel = quantize_dynamic(cpu_model, {nn.Linear, nn.GRU}, dtype=torch.qint8)
        with torch.no_grad():
            _ = qmodel(seq_t, ctx_t)
        return True, None
    except Exception as e:
        return False, str(e)


def plot_confusion(cm, title):
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(np.arange(NUM_CLASSES))
    ax.set_yticks(np.arange(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, str(cm[i][j]), ha='center', va='center', color='black', fontsize=7)
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


def train_v2_context_model_impl(train_bundle: dict, val_bundle: dict) -> dict:
    model = SeqContextFusionModel(
        num_bins=NUM_BINS,
        context_dim=train_bundle['context_dim'],
        num_classes=NUM_CLASSES,
        hidden_dim=48,
    )

    params = count_parameters(model)
    print(f"[{train_bundle['domain']}] model params: {params:,}")

    train_out = _train_model(
        model=model,
        train_loader=train_bundle['train_loader'],
        val_loader=val_bundle['val_loader'],
        device=DEVICE,
        max_epochs=MAX_EPOCHS_V2,
        patience=PATIENCE,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    val_probs, val_true, _ = _collect_probs(train_out['model'], val_bundle['val_loader'], DEVICE)
    unknown_threshold = _choose_unknown_threshold(val_probs, val_true, unknown_idx=6)

    train_out['unknown_threshold'] = float(unknown_threshold)
    train_out['params'] = int(params)
    train_out['domain'] = train_bundle['domain']
    return train_out


def evaluate_v2_impl(model_bundle: dict, test_bundle: dict) -> dict:
    model = model_bundle['model']
    threshold = float(model_bundle['unknown_threshold'])

    probs, y_true, regime_ids = _collect_probs(model, test_bundle['test_loader'], DEVICE)
    raw_pred = np.argmax(probs, axis=1)

    unknown_score = 1.0 - probs.max(axis=1)
    gated_pred = raw_pred.copy()
    gated_pred[unknown_score >= threshold] = 6

    raw_metrics = _metrics_from_preds(y_true, raw_pred, probs, regime_ids, CLASS_NAMES, REGIMES)
    gated_metrics = _metrics_from_preds(y_true, gated_pred, probs, regime_ids, CLASS_NAMES, REGIMES)

    seq_sample = test_bundle['normed']['test']['X_seq_n'][0]
    ctx_sample = test_bundle['normed']['test']['X_context_n'][0]

    latency_p95_ms = _measure_latency_p95_cpu_ms(model, seq_sample, ctx_sample)
    quant_ready, quant_error = _quantization_readiness(model, seq_sample, ctx_sample)

    edge_targets = V2_EDGE_TARGETS
    edge_pass = (
        model_bundle['params'] <= edge_targets['max_params']
        and latency_p95_ms <= edge_targets['p95_latency_ms_cpu_proxy']
        and bool(quant_ready) == bool(edge_targets['quantization_ready'])
    )

    return {
        'domain': model_bundle['domain'],
        'unknown_threshold': threshold,
        'raw_metrics': raw_metrics,
        'gated_metrics': gated_metrics,
        'edge': {
            'params': int(model_bundle['params']),
            'p95_latency_ms_cpu_proxy': float(latency_p95_ms),
            'quantization_ready': bool(quant_ready),
            'quantization_error': quant_error,
            'pass_edge_gate': bool(edge_pass),
        },
    }

# Bind scaffold aliases.
train_v2_context_model_impl = train_v2_context_model_impl
evaluate_v2_impl = evaluate_v2_impl


## 4. V2 Training by Domain


In [ ]:
# Train one domain model per signal domain
V2_MODELS = {}

for domain in V2_SIGNAL_DOMAINS:
    train_bundle = {
        'domain': domain,
        'train_loader': DOMAIN_BUNDLES[domain]['train_loader'],
        'context_dim': DOMAIN_BUNDLES[domain]['context_dim'],
    }
    val_bundle = {
        'domain': domain,
        'val_loader': DOMAIN_BUNDLES[domain]['val_loader'],
    }

    model_out = train_v2_context_model(train_bundle, val_bundle)
    ckpt_name = 'v2_downstream_seq_context_best.pt' if domain == 'downstream_rxmer' else 'v2_upstream_seq_context_best.pt'
    ckpt_path = CKPT_DIR / ckpt_name
    torch.save(model_out['model'].state_dict(), ckpt_path)
    print(f'[{domain}] saved checkpoint: {ckpt_path.resolve()}')

    V2_MODELS[domain] = model_out


## 5. V2 Evaluation and Edge Gates


In [ ]:
V2_RESULTS = {}

for domain in V2_SIGNAL_DOMAINS:
    test_bundle = {
        'domain': domain,
        'test_loader': DOMAIN_BUNDLES[domain]['test_loader'],
        'normed': DOMAIN_BUNDLES[domain]['normed'],
    }
    result = evaluate_v2(V2_MODELS[domain], test_bundle)
    V2_RESULTS[domain] = result

    print(f"[{domain}] threshold={result['unknown_threshold']:.3f}")
    print(f"[{domain}] gated macro F1={result['gated_metrics']['macro_f1']:.4f}")
    print(f"[{domain}] event_peak macro F1={result['gated_metrics']['event_peak_macro_f1']:.4f}")
    print(f"[{domain}] impulse F1={result['gated_metrics']['impulse_f1']:.4f}")
    print(f"[{domain}] unknown F1={result['gated_metrics']['unknown_f1']:.4f}")
    print(f"[{domain}] edge pass={result['edge']['pass_edge_gate']} | params={result['edge']['params']:,} | p95ms={result['edge']['p95_latency_ms_cpu_proxy']:.3f} | quant={result['edge']['quantization_ready']}")

plot_confusion(V2_RESULTS['downstream_rxmer']['gated_metrics']['confusion_matrix'], 'V2 Downstream Gated Confusion Matrix')
plot_confusion(V2_RESULTS['upstream_return']['gated_metrics']['confusion_matrix'], 'V2 Upstream Gated Confusion Matrix')


## 6. V2 Side-by-Side Domain Comparison


In [ ]:
legacy_public_event_peak = None
legacy_summary_path = METRICS_DIR / 'run_summary.json'
if legacy_summary_path.exists():
    try:
        with legacy_summary_path.open('r', encoding='utf-8') as f:
            legacy_summary = json.load(f)
        legacy_public_event_peak = legacy_summary.get('public', {}).get('event_peak_macro_f1')
    except Exception:
        legacy_public_event_peak = None

comparison_rows = []
for domain, model_name in [
    ('downstream_rxmer', 'downstream_seq_context'),
    ('upstream_return', 'upstream_seq_context'),
]:
    res = V2_RESULTS[domain]
    gm = res['gated_metrics']
    edge = res['edge']

    if legacy_public_event_peak is None:
        event_peak_delta_vs_public = float('nan')
        pass_quality = gm['macro_f1'] >= V2_ACCEPTANCE_TARGETS['macro_f1_min']
    else:
        event_peak_delta_vs_public = gm['event_peak_macro_f1'] - float(legacy_public_event_peak)
        pass_quality = (
            gm['macro_f1'] >= V2_ACCEPTANCE_TARGETS['macro_f1_min']
            and event_peak_delta_vs_public >= V2_ACCEPTANCE_TARGETS['event_peak_macro_f1_delta_vs_public_min']
        )

    row = {
        'domain': domain,
        'model_name': model_name,
        'params': edge['params'],
        'p95_latency_ms_cpu_proxy': edge['p95_latency_ms_cpu_proxy'],
        'quantization_ready': edge['quantization_ready'],
        'macro_f1': gm['macro_f1'],
        'event_peak_macro_f1': gm['event_peak_macro_f1'],
        'anomaly_auroc': gm['anomaly_auroc'],
        'impulse_f1': gm['impulse_f1'],
        'unknown_f1': gm['unknown_f1'],
        'unknown_threshold': res['unknown_threshold'],
        'event_peak_delta_vs_legacy_public': event_peak_delta_vs_public,
        'pass_edge_gate': edge['pass_edge_gate'],
        'pass_quality_targets': bool(pass_quality),
        'pass_all_gates': bool(edge['pass_edge_gate'] and pass_quality),
    }
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)

# Conform scaffold table fields.
v2_eval_template_df = comparison_df[[
    'model_name',
    'params',
    'p95_latency_ms_cpu_proxy',
    'quantization_ready',
    'macro_f1',
    'event_peak_macro_f1',
    'anomaly_auroc',
    'impulse_f1',
    'unknown_f1',
    'pass_edge_gate',
]].copy()

comparison_df


In [ ]:
v2_eval_template_df


## 7. Save Metrics and Run Summary (v2)


In [ ]:
def to_builtin(x):
    if isinstance(x, dict):
        return {k: to_builtin(v) for k, v in x.items()}
    if isinstance(x, list):
        return [to_builtin(v) for v in x]
    if isinstance(x, tuple):
        return [to_builtin(v) for v in x]
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, np.ndarray):
        return x.tolist()
    return x

# Keep writing/validating the contract.
with v2_contract_path.open('w', encoding='utf-8') as f:
    json.dump(v2_contract, f, indent=2)

v2_ds_metrics_path = METRICS_DIR / 'v2_downstream_metrics.json'
v2_us_metrics_path = METRICS_DIR / 'v2_upstream_metrics.json'
v2_comparison_csv_path = METRICS_DIR / 'v2_model_comparison.csv'
v2_summary_path = METRICS_DIR / 'v2_run_summary.json'

with v2_ds_metrics_path.open('w', encoding='utf-8') as f:
    json.dump(to_builtin(V2_RESULTS['downstream_rxmer']), f, indent=2)

with v2_us_metrics_path.open('w', encoding='utf-8') as f:
    json.dump(to_builtin(V2_RESULTS['upstream_return']), f, indent=2)

comparison_df.to_csv(v2_comparison_csv_path, index=False)

v2_run_summary = {
    'config': {
        'seed': SEED,
        'num_bins': NUM_BINS,
        'time_steps': TIME_STEPS,
        'num_classes': NUM_CLASSES,
        'class_names': CLASS_NAMES,
        'regimes': REGIMES,
        'signal_domains': V2_SIGNAL_DOMAINS,
        'node_context_features': V2_NODE_CONTEXT_FEATURES,
        'train_samples': TRAIN_SAMPLES,
        'val_samples': VAL_SAMPLES,
        'test_samples': TEST_SAMPLES,
        'device': str(DEVICE),
        'fast_dev_mode': FAST_DEV_MODE,
    },
    'generation_seconds_by_domain': v2_generation_seconds,
    'edge_targets': V2_EDGE_TARGETS,
    'acceptance_targets': V2_ACCEPTANCE_TARGETS,
    'legacy_public_event_peak_reference': legacy_public_event_peak,
    'domains': {
        k: {
            'unknown_threshold': V2_RESULTS[k]['unknown_threshold'],
            'gated_macro_f1': V2_RESULTS[k]['gated_metrics']['macro_f1'],
            'gated_event_peak_macro_f1': V2_RESULTS[k]['gated_metrics']['event_peak_macro_f1'],
            'gated_impulse_f1': V2_RESULTS[k]['gated_metrics']['impulse_f1'],
            'gated_unknown_f1': V2_RESULTS[k]['gated_metrics']['unknown_f1'],
            'anomaly_auroc': V2_RESULTS[k]['gated_metrics']['anomaly_auroc'],
            'params': V2_RESULTS[k]['edge']['params'],
            'p95_latency_ms_cpu_proxy': V2_RESULTS[k]['edge']['p95_latency_ms_cpu_proxy'],
            'quantization_ready': V2_RESULTS[k]['edge']['quantization_ready'],
            'pass_edge_gate': V2_RESULTS[k]['edge']['pass_edge_gate'],
        }
        for k in V2_SIGNAL_DOMAINS
    },
    'comparison': to_builtin(comparison_rows),
    'overall_pass_all_gates': bool(comparison_df['pass_all_gates'].all()),
}

with v2_summary_path.open('w', encoding='utf-8') as f:
    json.dump(to_builtin(v2_run_summary), f, indent=2)

print('Saved v2 artifacts:')
print('-', dataset_path.resolve())
print('-', v2_ds_metrics_path.resolve())
print('-', v2_us_metrics_path.resolve())
print('-', v2_comparison_csv_path.resolve())
print('-', v2_summary_path.resolve())
print('-', v2_contract_path.resolve())


## Notes

- This notebook now executes the v2 domain-aware sequence+context pipeline as the primary runnable flow.
- Existing v1 narrative sections were retained for historical context, but v2 artifacts are written under `v2_*` names.
